# Module 2 — KG Construction & Named Entity Recognition

**Goal:** extract medical entities (Disease, Drug, Symptom, Procedure, Anatomy) from patient summaries using scispaCy, link them to UMLS concepts, and load the resulting entity-relation graph into a Neo4j Aura database.

This is the Colab port of the local version of this notebook. The logic and graph schema are unchanged — same `(:Patient)-[:MENTIONS]->(:Concept)` structure, same two-phase approach (raw NER first, then UMLS linking) — only the environment setup and the Neo4j connection details differ, since Colab needs its own package installs each session and Aura is used here instead of a local Neo4j Desktop instance.

Before running: this notebook reads the raw CSV directly from Drive, independently of Module 1's chunked output, so it can be run on its own once `00_setup.ipynb` has confirmed the dataset is present.

In [1]:
import time

start_time = time.time()

## 1. Install packages

`nmslib` is required by scispaCy's UMLS EntityLinker for approximate nearest-neighbour search over concept embeddings, and is not pulled in automatically by a plain `pip install scispacy`.

The `en_core_sci_sm` model release pins an old spaCy version with no wheel for Colab's current Python, so `scispacy` is installed on its own first, then the model with `--no-deps`, avoiding the source-build failure that pinning would otherwise trigger.

In [2]:
!pip install -q nmslib-metabrainz==2.1.3
!pip install -q --no-deps scispacy
!pip install -q conllu pysbd scikit-learn scipy joblib
!pip install -q neo4j

In [3]:
!pip install -q --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

  Preparing metadata (setup.py) ... done


## 2. Imports, model config patch, and Neo4j connection

The config patch below converts quoted `"True"`/`"False"` strings in the model's `config.cfg` into real booleans — a known incompatibility between this older model release and Colab's newer spaCy, otherwise loading raises a `ConfigValidationError`.

Connection details use Colab Secrets rather than a hardcoded password. `NEO4J_USERNAME` is read from secrets too, rather than assumed to be `"neo4j"`, since the current Aura instance issues an instance-ID-style username.

In [4]:
import os
import glob
import re
import pandas as pd
from tqdm.auto import tqdm

import en_core_sci_sm
from neo4j import GraphDatabase
from google.colab import drive, userdata

drive.mount('/content/drive')

# --- Patch the model config before loading ---
pkg_dir = os.path.dirname(en_core_sci_sm.__file__)
config_paths = glob.glob(os.path.join(pkg_dir, "**", "config.cfg"), recursive=True)
for path in config_paths:
    with open(path, "r") as f:
        content = f.read()
    fixed = re.sub(r'=\s*"True"', "= true", content)
    fixed = re.sub(r'=\s*"False"', "= false", fixed)
    if fixed != content:
        with open(path, "w") as f:
            f.write(fixed)
        print("Patched:", path)

nlp = en_core_sci_sm.load()
print("scispaCy model loaded:", nlp.meta["name"])

# --- Neo4j Aura connection ---
NEO4J_URI = userdata.get('NEO4J_URI')
NEO4J_USERNAME = userdata.get('NEO4J_USERNAME')
NEO4J_PASSWORD = userdata.get('NEO4J_PASSWORD')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

def test_connection():
    with driver.session() as session:
        result = session.run("RETURN 'Connected from Python!' AS message")
        for record in result:
            print(record["message"])

test_connection()

/usr/local/lib/python3.13/dist-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_core_sci_sm' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.16). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.13/dist-packages/spacy/language.py:2235: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


scispaCy model loaded: core_sci_sm
Connected from Python!


## 3. Quick sanity check on the loaded model

In [5]:
test_text = "Patient was started on Metformin for type 2 diabetes and later developed acute kidney injury."
doc = nlp(test_text)
for ent in doc.ents:
    print(ent.text, "->", ent.label_)

Patient -> ENTITY
Metformin -> ENTITY
type 2 diabetes -> ENTITY
acute kidney injury -> ENTITY


## 4. Load a patient subset

`SUBSET_SIZE = 50` matches the subset already used for the verified knowledge graph reported in the thesis (50 patients, 2,297 UMLS concept nodes, 4,376 MENTIONS relationships). This can be scaled up independently of Module 1's ingestion subset once the pipeline is reconfirmed end-to-end on Colab.

In [6]:
PROJECT_ROOT = "/content/drive/MyDrive/ClinicalTrust"
CSV_PATH = os.path.join(PROJECT_ROOT, "data/raw/PMC-Patients.csv")
TEXT_COLUMN = "patient"
ID_COLUMN = "patient_id"

SUBSET_SIZE = 50  # matches the verified thesis-reported graph; raise once reconfirmed on Colab

chunks_iter = pd.read_csv(CSV_PATH, chunksize=SUBSET_SIZE)
df = next(chunks_iter)
df = df.dropna(subset=[TEXT_COLUMN]).reset_index(drop=True)

print(df.shape)
df[[ID_COLUMN, TEXT_COLUMN]].head()

(50, 10)


,patient_id,patient
0,0,This 60-year-old male was hospitalized due to ...
1,1,A 39-year-old man was hospitalized due to an i...
2,2,One week after a positive COVID-19 result this...
3,3,This 69-year-old male was admitted to the ICU ...
4,4,This 57-year-old male was admitted to the ICU ...


## 5. Run NER on each patient summary

scispaCy processes text in batches efficiently via `nlp.pipe()`. For each patient, the list of entity spans found is collected.

In [7]:
entity_records = []

texts = df[TEXT_COLUMN].tolist()
patient_ids = df[ID_COLUMN].tolist()

for patient_id, doc in tqdm(zip(patient_ids, nlp.pipe(texts, batch_size=8)), total=len(texts)):
    for ent in doc.ents:
        entity_records.append({
            "patient_id": patient_id,
            "entity_text": ent.text,
            "start_char": ent.start_char,
            "end_char": ent.end_char
        })

entities_df = pd.DataFrame(entity_records)
print(f"Extracted {len(entities_df)} entity mentions across {len(df)} patients")
entities_df.head(10)

  0%|          | 0/50 [00:00<?, ?it/s]

Extracted 6141 entity mentions across 50 patients


,patient_id,entity_text,start_char,end_char
0,0,male,17,21
1,0,hospitalized,26,38
2,0,moderate,46,54
3,0,ARDS,55,59
4,0,COVID-19,65,73
5,0,symptoms,79,87
6,0,fever,91,96
7,0,dry cough,98,107
8,0,dyspnea,113,120
9,0,difficulties,145,157


## 6. UMLS linking (via scispaCy's built-in linker)

scispaCy ships an entity linker that resolves extracted spans to UMLS Concept Unique Identifiers (CUIs) using its own bundled knowledge base — this does not require a personal NLM API key for this step, since it uses a local, pre-built UMLS index. The underlying data file is a larger download (roughly 1GB) the first time it runs in a session, and stays cached for the rest of that session.

In [8]:
from scispacy.linking import EntityLinker

# Add the linker to the pipeline (first run downloads ~1GB knowledge base, cached after)
nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})
linker = nlp.get_pipe("scispacy_linker")

print("UMLS linker loaded.")

https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/linkers/2023-04-23/umls/tfidf_vectors_sparse.npz not found in cache, downloading to /tmp/tmpwtd63z7q


100%|██████████| 492M/492M [00:10<00:00, 47.1MiB/s]


Finished download, copying /tmp/tmpwtd63z7q to cache at /root/.scispacy/datasets/2b79923846fb52e62d686f2db846392575c8eb5b732d9d26cd3ca9378c622d40.87bd52d0f0ee055c1e455ef54ba45149d188552f07991b765da256a1b512ca0b.tfidf_vectors_sparse.npz
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/linkers/2023-04-23/umls/nmslib_index.bin not found in cache, downloading to /tmp/tmp4qz1jage


100%|██████████| 724M/724M [00:27<00:00, 28.1MiB/s]


Finished download, copying /tmp/tmp4qz1jage to cache at /root/.scispacy/datasets/7e8e091ec80370b87b1652f461eae9d926e543a403a69c1f0968f71157322c25.6d801a1e14867953e36258b0e19a23723ae84b0abd2a723bdd3574c3e0c873b4.nmslib_index.bin
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/linkers/2023-04-23/umls/tfidf_vectorizer.joblib not found in cache, downloading to /tmp/tmpevqexi35


...

Finished download, copying /tmp/tmpxc54eno2 to cache at /root/.scispacy/datasets/6238f505f56aca33290aab44097f67dd1b88880e3be6d6dcce65e56e9255b7d4.d7f77b1629001b40f1b1bc951f3a890ff2d516fb8fbae3111b236b31b33d6dcf.concept_aliases.json
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/kbs/2023-04-23/umls_2022_ab_cat0129.jsonl not found in cache, downloading to /tmp/tmpo_e8lkb1


100%|██████████| 628M/628M [00:25<00:00, 25.6MiB/s]


Finished download, copying /tmp/tmpo_e8lkb1 to cache at /root/.scispacy/datasets/d5e593bc2d8adeee7754be423cd64f5d331ebf26272074a2575616be55697632.0660f30a60ad00fffd8bbf084a18eb3f462fd192ac5563bf50940fc32a850a3c.umls_2022_ab_cat0129.jsonl
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/umls_semantic_type_tree.tsv not found in cache, downloading to /tmp/tmph5r2wf0o


100%|██████████| 4.26k/4.26k [00:00<00:00, 10.7MiB/s]

Finished download, copying /tmp/tmph5r2wf0o to cache at /root/.scispacy/datasets/21a1012c532c3a431d60895c509f5b4d45b0f8966c4178b892190a302b21836f.330707f4efe774134872b9f77f0e3208c1d30f50800b3b39a6b8ec21d9adf1b7.umls_semantic_type_tree.tsv
UMLS linker loaded.


In [9]:
linked_records = []

for patient_id, doc in tqdm(zip(patient_ids, nlp.pipe(texts, batch_size=8)), total=len(texts)):
    for ent in doc.ents:
        if ent._.kb_ents:
            # Take the top-ranked UMLS candidate match
            cui, score = ent._.kb_ents[0]
            umls_entity = linker.kb.cui_to_entity[cui]
            linked_records.append({
                "patient_id": patient_id,
                "entity_text": ent.text,
                "cui": cui,
                "canonical_name": umls_entity.canonical_name,
                "semantic_types": umls_entity.types,
                "confidence": round(score, 3)
            })

linked_df = pd.DataFrame(linked_records)
print(f"Linked {len(linked_df)} entity mentions to UMLS concepts")
linked_df.head(10)

  0%|          | 0/50 [00:00<?, ?it/s]

Linked 5675 entity mentions to UMLS concepts


,patient_id,entity_text,cui,canonical_name,semantic_types,confidence
0,0,male,C0086582,Males,[T032],0.995
1,0,hospitalized,C0701159,Patient in hospital (finding),[T033],0.983
2,0,moderate,C0205081,Moderate (severity modifier),[T080],0.983
3,0,ARDS,C0035222,"Respiratory Distress Syndrome, Adult",[T047],0.972
4,0,COVID-19,C5203670,COVID19 (disease),[T047],0.979
5,0,symptoms,C0683368,Symptoms aspect,[T169],0.977
6,0,fever,C0015967,Fever,[T184],0.962
7,0,dry cough,C0850149,Dry cough,[T184],0.964
8,0,dyspnea,C0013404,Dyspnea,[T184],0.984
9,0,difficulties,C1299586,Has difficulty doing (qualifier value),[T033],0.985


## 7. Load entities and relationships into Neo4j

For each patient, a `Patient` node and `Concept` nodes for each linked UMLS entity are created, connected by a `MENTIONS` relationship. This is the schema Module 3 queries against.

In [10]:
def load_patient_entities(tx, patient_id, entities):
    tx.run(
        "MERGE (p:Patient {patient_id: $patient_id})",
        patient_id=patient_id
    )
    for e in entities:
        tx.run(
            """
            MERGE (c:Concept {cui: $cui})
            ON CREATE SET c.canonical_name = $canonical_name, c.semantic_types = $semantic_types
            WITH c
            MATCH (p:Patient {patient_id: $patient_id})
            MERGE (p)-[r:MENTIONS]->(c)
            SET r.entity_text = $entity_text, r.confidence = $confidence
            """,
            cui=e["cui"],
            canonical_name=e["canonical_name"],
            semantic_types=e["semantic_types"],
            patient_id=patient_id,
            entity_text=e["entity_text"],
            confidence=e["confidence"]
        )

with driver.session() as session:
    for patient_id in tqdm(linked_df["patient_id"].unique()):
        patient_entities = linked_df[linked_df["patient_id"] == patient_id].to_dict("records")
        session.execute_write(load_patient_entities, patient_id, patient_entities)

print("Loaded patient-entity graph into Neo4j.")

  0%|          | 0/50 [00:00<?, ?it/s]

Loaded patient-entity graph into Neo4j.


## 8. Verify the graph

Check node/relationship counts, and spot-check one patient's linked concepts.

In [11]:
with driver.session() as session:
    counts = session.run(
        """
        MATCH (p:Patient) WITH count(p) AS patients
        MATCH (c:Concept) WITH patients, count(c) AS concepts
        MATCH ()-[r:MENTIONS]->() WITH patients, concepts, count(r) AS mentions
        RETURN patients, concepts, mentions
        """
    )
    for record in counts:
        print(record)

    sample = session.run(
        """
        MATCH (p:Patient)-[r:MENTIONS]->(c:Concept)
        RETURN p.patient_id AS patient, c.canonical_name AS concept, r.entity_text AS mentioned_as
        LIMIT 10
        """
    )
    print("\nSample results:")
    for record in sample:
        print(dict(record))

driver.close()

<Record patients=50 concepts=2297 mentions=4376>

Sample results:
{'patient': 0, 'concept': 'Males', 'mentioned_as': 'male'}
{'patient': 0, 'concept': 'Patient in hospital (finding)', 'mentioned_as': 'hospitalized'}
{'patient': 0, 'concept': 'Moderate (severity modifier)', 'mentioned_as': 'moderate'}
{'patient': 0, 'concept': 'Respiratory Distress Syndrome, Adult', 'mentioned_as': 'ARDS'}
{'patient': 0, 'concept': 'COVID19 (disease)', 'mentioned_as': 'COVID-19'}
{'patient': 0, 'concept': 'Symptoms aspect', 'mentioned_as': 'symptoms'}
{'patient': 0, 'concept': 'Fever', 'mentioned_as': 'fever'}
{'patient': 0, 'concept': 'Dry cough', 'mentioned_as': 'dry cough'}
{'patient': 0, 'concept': 'Dyspnea', 'mentioned_as': 'dyspnea'}
{'patient': 0, 'concept': 'Has difficulty doing (qualifier value)', 'mentioned_as': 'difficulty'}


## Next steps

1. If this reproduces the expected figures (50 patients, ~2,297 concepts, ~4,376 mentions), `SUBSET_SIZE` can be raised gradually (200 -> 1000 -> full) and re-run, watching runtime as in Module 1. Small differences from the exact local figures are expected since UMLS linking has some non-determinism at the margin (near-threshold candidate scores).
2. DrugBank supplementary edges (drug-drug interactions) can be added later as an extra loading step once the core UMLS-linked graph is stable.
3. Manually validate a random sample of entity-CUI mappings (per the proposal's Risk R5) — pull roughly 100 rows from `linked_df` and spot-check whether `canonical_name` matches what a clinician would expect for that `entity_text`.
4. Module 3 will query this graph (via Cypher traversal) alongside Module 1's FAISS index for dual-pathway retrieval.
5. Save this notebook into `ClinicalTrust/notebooks/` on Drive alongside Module 1.

In [12]:
end_time = time.time()

execution_time = end_time - start_time

print(f"Total execution time: {execution_time:.2f} seconds")
print(f"Total execution time: {execution_time/60:.2f} minutes")

Total execution time: 1264.43 seconds
Total execution time: 21.07 minutes
